# 01. Literały wbudowane

## Cele

Po tym notebooku potrafisz:

* odczytać typ literału liczbowego z jego zapisu: systemu liczbowego, separatorów i sufiksu;
* wskazać typ literału znakowego i napisowego oraz działanie sekwencji ucieczki (`\n`, `\t`, `\\`);
* zapisać surowy napis `R"(...)"` (także z własnym ogranicznikiem) i wiedzieć, kiedy go użyć;
* korzystać z sufiksów biblioteki standardowej: `s`, `sv` i sufiksów `std::chrono` (`h`, `min`, `s`, `ms`…).

Fragmenty kodu w tym notebooku są samodzielnymi programami — każdy był skompilowany i uruchomiony
(GCC i Clang, `-std=c++20 -Wall -Wextra -Wpedantic`), a wynik podajemy w komentarzu. Wiele z nich sprawdza
typy przez `static_assert(std::is_same_v<…>)`: jeśli program się kompiluje, typ jest taki, jak w komentarzu.

## Literały całkowite

Liczbę całkowitą można zapisać w czterech systemach: dziesiętnym (`42`), ósemkowym — z wiodącym zerem (`052`),
szesnastkowym (`0x2A`) i, od C++14, dwójkowym (`0b101010`). Pojedyncze apostrofy (C++14) oddzielają grupy cyfr
i nie zmieniają wartości: `1'000'000` to milion
([cppreference: Integer literal](https://en.cppreference.com/cpp/language/integer_literal),
[standard: lex.icon](https://eel.is/c++draft/lex.icon#tab:lex.icon.type)).

**Typ** literału całkowitego to pierwszy typ z listy, w którym mieści się wartość. Lista zależy od sufiksu
i od systemu liczbowego
([cppreference: The type of the literal](https://en.cppreference.com/cpp/language/integer_literal#The_type_of_the_literal)):

| Sufiks | Literał dziesiętny | Literał ósemkowy, szesnastkowy, dwójkowy |
|---|---|---|
| brak | `int`, `long`, `long long` | `int`, `unsigned int`, `long`, `unsigned long`, `long long`, `unsigned long long` |
| `u` / `U` | `unsigned int`, `unsigned long`, `unsigned long long` | tak samo |
| `l` / `L` | `long`, `long long` | `long`, `unsigned long`, `long long`, `unsigned long long` |
| `ll` / `LL` | `long long` | `long long`, `unsigned long long` |

Sufiksy można łączyć (`ul`, `ull`), a wielkość liter nie ma znaczenia.

```cpp
#include <iostream>
#include <type_traits>

int main() {
    // the same value written in four bases
    static_assert(42 == 052 && 42 == 0x2A && 42 == 0b10'1010);
    // digit separators are ignored
    static_assert(1'000'000 == 1000000);

    static_assert(std::is_same_v<decltype(42), int>);
    static_assert(std::is_same_v<decltype(42u), unsigned int>);
    static_assert(std::is_same_v<decltype(42l), long>);
    static_assert(std::is_same_v<decltype(42ull), unsigned long long>);

    // too big for int: the next type on the list (long on 64-bit Linux and macOS)
    auto big = 10'000'000'000;
    std::cout << big << ' ' << sizeof(big) << '\n';  // 10000000000 8

    // a leading zero means octal!
    int code = 0123;
    std::cout << code << '\n';  // 83
}
```

Rozmiar typów zależy od platformy: na 64-bitowym Linuksie i macOS `long` ma 64 bity, a w Windows tylko 32,
więc tam ten sam literał `10'000'000'000` jest typu `long long`. Wartość jest ta sama — zmienia się typ.

## Literały zmiennoprzecinkowe

Literał z kropką albo wykładnikiem (`2.5`, `1e3`, `2.5e-3`) jest typu `double`. Sufiks `f`/`F` daje `float`,
a `l`/`L` — `long double`
([cppreference: Floating-point literal](https://en.cppreference.com/cpp/language/floating_literal#Syntax),
[standard: lex.fcon](https://eel.is/c++draft/lex.fcon#tab:lex.fcon.type)).
Typ literału decyduje o tym, jak liczy wyrażenie: dwie liczby całkowite dzielą się całkowicie.

```cpp
#include <iostream>
#include <type_traits>

int main() {
    static_assert(std::is_same_v<decltype(2.5), double>);
    static_assert(std::is_same_v<decltype(2.5f), float>);
    static_assert(std::is_same_v<decltype(2.5L), long double>);
    static_assert(1e3 == 1000.0);

    std::cout << 7 / 2 << ' ' << 7.0 / 2 << ' ' << 7 / 2.0f << '\n';  // 3 3.5 3.5
}
```

## Literały znakowe i napisowe

Literał znakowy w apostrofach (`'a'`) jest typu `char`. Przedrostek zmienia typ i kodowanie:
`u8'a'` to `char8_t` (C++20), `u'a'` — `char16_t`, `U'a'` — `char32_t`, `L'a'` — `wchar_t`
([standard: lex.ccon](https://eel.is/c++draft/lex.ccon#tab:lex.ccon.literal)).

Literał napisowy w cudzysłowie to **tablica** stałych znaków zakończona znakiem `'\0'`: `"Ala"` ma typ
`const char[4]`. Przedrostki działają tak samo jak dla znaków (`u8"…"` to tablica `const char8_t`)
([cppreference: String literal](https://en.cppreference.com/cpp/language/string_literal#Explanation)).
Tablica w wielu miejscach zamienia się we wskaźnik na pierwszy element — dlatego `auto s = "Ala";`
daje `const char*`, a nie `std::string`.

Wewnątrz zwykłego literału ukośnik `\` zaczyna **sekwencję ucieczki**: `\n` to nowa linia, `\t` — tabulator,
`\"` — cudzysłów, a `\\` — sam ukośnik
([cppreference: Escape sequences](https://en.cppreference.com/cpp/language/escape)).
Sąsiednie literały napisowe kompilator łączy w jeden — tak zapisuje się długie teksty w kilku liniach
([cppreference: Concatenation](https://en.cppreference.com/cpp/language/string_literal#Concatenation)).

```cpp
#include <iostream>
#include <type_traits>

int main() {
    static_assert(std::is_same_v<decltype('a'), char>);
    static_assert(std::is_same_v<decltype(u8'a'), char8_t>);
    static_assert(std::is_same_v<decltype(U'a'), char32_t>);

    const char (&array)[4] = "Ala";  // binds only because the type is exactly const char[4]
    auto pointer = "Ala";            // the array decays to a pointer
    static_assert(std::is_same_v<decltype(pointer), const char*>);
    static_assert(sizeof("Ala") == 4);  // three letters + '\0'

    // adjacent literals form one string
    const char* usage = "Usage: report [options]\n"
                        "  -o FILE\twrite to FILE\n";
    std::cout << array << ' ' << pointer << '\n' << usage;
    std::cout << "\"quoted\" C:\\temp\n";  // prints: "quoted" C:\temp
}
```

Dwa literały napisowe to dwie tablice, więc nie da się ich dodać operatorem `+`:

```cpp
#include <string>

std::string greeting = "Ala" + " i Ola";  // error: invalid operands 'const char[4]' and 'const char[7]'
```

Rozwiązaniem jest sufiks `s` z biblioteki standardowej — o nim w sekcji „Literały biblioteki standardowej”.

## Surowe napisy

W zwykłym literale każdy `\` trzeba podwoić. Przy ścieżkach Windows i wyrażeniach regularnych łatwo o pomyłkę,
a pomyłka bywa cicha: `\n` i `\t` są poprawnymi sekwencjami ucieczki, więc kompilator nie zgłosi nawet
ostrzeżenia.

**Surowy napis** (*raw string literal*, C++11) ma przedrostek `R` i treść w nawiasach: `R"(...)"`.
Wszystko między `(` a `)` trafia do napisu dosłownie — sekwencje ucieczki nie są przetwarzane, a nowe
linie w kodzie stają się znakami nowej linii w napisie
([cppreference: Raw string literals](https://en.cppreference.com/cpp/language/string_literal#Raw_string_literals),
[Microsoft Learn: Raw string literals](https://learn.microsoft.com/en-us/cpp/cpp/string-and-character-literals-cpp#raw-string-literals-c11)).

```cpp
#include <iostream>
#include <string_view>

int main() {
    std::string_view escaped = "D:\\nauka\\testy";
    std::string_view raw = R"(D:\nauka\testy)";
    std::cout << raw << ' ' << (escaped == raw) << '\n';  // D:\nauka\testy 1

    std::string_view broken = "D:\nauka\testy";  // \n is a newline, \t is a tab
    std::cout << broken.size() << " vs " << raw.size() << '\n';  // 12 vs 14
}
```

Pełna składnia to `R"ogranicznik(treść)ogranicznik"`. **Ogranicznik** jest opcjonalny (do 16 znaków, bez
spacji, nawiasów i `\`), a **nawiasy są obowiązkowe**
([standard: lex.string](https://eel.is/c++draft/lex.string#2)). Ogranicznik jest potrzebny, gdy treść
zawiera sekwencję `)"`, która bez niego zakończyłaby napis:

```cpp
#include <iostream>
#include <regex>
#include <string_view>

int main() {
    // the text contains )" -- the delimiter "json" marks the real end
    constexpr std::string_view reply = R"json({"status": "ok (200)"})json";
    std::cout << reply << '\n';  // {"status": "ok (200)"}

    // multi-line text: line breaks (and indentation!) become part of the string
    constexpr std::string_view query = R"(SELECT name
FROM students
WHERE grade = 5)";
    std::cout << query << '\n';

    // a regular expression without doubled backslashes: YYYY-MM-DD
    std::regex date(R"((\d{4})-(\d{2})-(\d{2}))");
    std::cout << std::regex_match("2026-11-09", date) << '\n';  // 1
}
```

Bez nawiasów kompilator czyta tekst po cudzysłowie jako ogranicznik i natrafia na niedozwolony znak:

```cpp
const char* text = R"Ala ma kota";  // error: invalid character ' ' in raw string delimiter
```

## Literały biblioteki standardowej

Od C++14 biblioteka standardowa definiuje własne sufiksy. Najważniejsze z nich
([cppreference: operator""s](https://en.cppreference.com/cpp/string/basic_string/operator%22%22s#Notes),
[operator""sv](https://en.cppreference.com/cpp/string/basic_string_view/operator%22%22sv#Notes),
[chrono literals](https://en.cppreference.com/cpp/chrono/duration#Literals)):

| Zapis | Typ | Przestrzeń nazw | Nagłówek |
|---|---|---|---|
| `"tekst"s` | `std::string` | `std::literals::string_literals` | `<string>` |
| `"tekst"sv` | `std::string_view` | `std::literals::string_view_literals` | `<string_view>` |
| `2h`, `30min`, `10s`, `250ms`, `5us`, `7ns` | `std::chrono::hours`, `minutes`, `seconds`, `milliseconds`… | `std::literals::chrono_literals` | `<chrono>` |
| `2i` ([cppreference](https://en.cppreference.com/cpp/numeric/complex/operator%22%22i#Notes)) | `std::complex<double>` | `std::literals::complex_literals` | `<complex>` |

Sufiksy trzeba wprowadzić dyrektywą `using namespace`. Przestrzenie `literals` i `…_literals` są *inline*,
więc działa każdy z zapisów: `using namespace std::literals;` (wszystkie sufiksy),
`using namespace std::string_literals;` albo `using namespace std::literals::string_literals;` (tylko jedna grupa).

```cpp
#include <iostream>
#include <string>
#include <string_view>
#include <type_traits>

using namespace std::literals;

int main() {
    auto name = "Ola"s;   // std::string
    auto view = "Ola"sv;  // std::string_view
    auto ptr = "Ola";     // const char*
    static_assert(std::is_same_v<decltype(name), std::string>);
    static_assert(std::is_same_v<decltype(view), std::string_view>);
    static_assert(std::is_same_v<decltype(ptr), const char*>);

    std::cout << name + " i Ala" << '\n';  // Ola i Ala

    // the length comes from the literal, so an embedded '\0' is kept
    std::cout << "a\0b"s.size() << " vs " << std::string("a\0b").size() << '\n';  // 3 vs 1
}
```

Sufiksy `std::chrono` tworzą czas trwania (`std::chrono::duration`) we właściwej jednostce. Suma `1h + 30min`
ma typ `std::chrono::minutes` — wynik dostaje jednostkę, w której oba składniki da się zapisać dokładnie.
Przypisanie do dokładniejszej jednostki (minuty → sekundy) działa bez rzutowania, bo nie gubi informacji,
a zamiana na jednostkę mniej dokładną wymaga `duration_cast`, który obcina
([cppreference: duration_cast](https://en.cppreference.com/cpp/chrono/duration/duration_cast#Notes)).

```cpp
#include <chrono>
#include <iostream>
#include <thread>
#include <type_traits>

using namespace std::literals;

int main() {
    auto lesson = 1h + 30min;
    static_assert(std::is_same_v<decltype(lesson), std::chrono::minutes>);
    std::cout << lesson.count() << " min\n";  // 90 min

    std::chrono::seconds pause = 2min;  // exact: OK without a cast
    std::cout << pause.count() << " s\n";  // 120 s

    auto whole = std::chrono::duration_cast<std::chrono::minutes>(150s);
    std::cout << whole.count() << " min\n";  // 2 min (truncated)

    std::this_thread::sleep_for(100ms);  // the unit is part of the type
}
```

**Ten sam sufiks `s` występuje dwa razy**: w `string_literals` (napis → `std::string`) i w `chrono_literals`
(liczba → sekundy). Nie ma konfliktu, bo o wyborze decyduje rodzaj literału: sufiks czasu dotyczy liczb,
a sufiks napisów — literałów napisowych. Po `using namespace std::literals;` dostępne są oba
([standard: time.duration.literals](https://eel.is/c++draft/time.duration.literals#8),
[cppreference: chrono operator""s](https://en.cppreference.com/cpp/chrono/operator%22%22s#Notes) —
„`10s` is ten seconds, but `"10"s` is a two-character string”).

```cpp
#include <chrono>
#include <string>
#include <type_traits>

using namespace std::literals;

static_assert(std::is_same_v<decltype(10s), std::chrono::seconds>);
static_assert(std::is_same_v<decltype("10"s), std::string>);

int main() {}
```

## Ciekawostki dla dociekliwych

### `0x80000000` i `2147483648` to różne typy

To ta sama wartość (2³¹), ale według tabeli typów z początku notebooka literał dziesiętny nigdy nie staje się
`unsigned` — przechodzi z `int` do `long`, a szesnastkowy trafia najpierw do `unsigned int`. Do tego
**w C++ nie ma ujemnych literałów**: `-2147483648` to operator minus zastosowany do literału `2147483648`
([cppreference: Integer literal — Notes](https://en.cppreference.com/cpp/language/integer_literal#Notes)).
Stąd zaskakujący wynik dla `-0x80000000`: minus działa na wartości `unsigned int`, więc wynik jest dodatni.

```cpp
#include <type_traits>

// GCC and Clang on 64-bit Linux and macOS (int: 32 bits, long: 64 bits)
static_assert(std::is_same_v<decltype(2147483647), int>);
static_assert(std::is_same_v<decltype(2147483648), long>);          // decimal: int -> long
static_assert(std::is_same_v<decltype(0x80000000), unsigned int>);  // hex: int -> unsigned int
static_assert(std::is_same_v<decltype(-2147483648), long>);         // unary minus on a long
static_assert(-0x80000000 == 2147483648);                           // unary minus on an unsigned int!

int main() {}
```

Tak mówi standard i tak działają GCC i Clang. Dokumentacja Microsoftu do ostrzeżenia C4146 opisuje
`2147483648` jako `unsigned int` w MSVC — tego zachowania nie sprawdzaliśmy
([Microsoft Learn: C4146](https://learn.microsoft.com/en-us/cpp/error-messages/compiler-warnings/compiler-warning-level-2-c4146)).

### C++20 zmienił typ `u8"…"`

Do C++17 literał `u8"tekst"` był tablicą `const char`. C++20 wprowadził osobny typ `char8_t` i od tej pory
`u8"tekst"` to tablica `const char8_t`, a `u8'x'` — `char8_t`
([P0482R6](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2018/p0482r6.html#proposal)). Kod, który
przypisywał `u8"…"` do `const char*`, przestał się kompilować:

```cpp
const char* text = u8"tekst";  // C++17: OK; C++20: error (const char8_t[6] -> const char*)
```

Usunięto też wypisywanie `char8_t` przez `std::cout` — inaczej `std::cout << u8'x'` wypisałby liczbę `120`,
a `std::cout << u8"text"` adres w pamięci
([P1423R3](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2019/p1423r3.html#option7)).

### `"\d"` w wyrażeniu regularnym

`\d` to w wyrażeniach regularnych „dowolna cyfra”
([cppreference: ECMAScript — Character classes](https://en.cppreference.com/cpp/regex/ecmascript#Character_classes)),
ale w C++ nie jest sekwencją ucieczki. Zapis `std::regex("\d+")` kompiluje się tylko z ostrzeżeniem
`unknown escape sequence '\d'` (GCC i Clang), a napis zawiera `d+` — wzorzec dopasuje więc `"ddd"`, a nie `"123"`.
Poprawnie: `"\\d+"` albo `R"(\d+)"`.

```cpp
#include <iostream>
#include <regex>

int main() {
    std::cout << std::regex_match("123", std::regex("\\d+")) << ' '
              << std::regex_match("123", std::regex(R"(\d+)")) << '\n';  // 1 1
}
```

## Linki i materiały

* [cppreference: Literals](https://en.cppreference.com/cpp/language/expressions#Literals) — przegląd wszystkich rodzajów literałów
* [cppreference: Integer literal](https://en.cppreference.com/cpp/language/integer_literal),
  [Floating-point literal](https://en.cppreference.com/cpp/language/floating_literal),
  [String literal](https://en.cppreference.com/cpp/language/string_literal),
  [Escape sequences](https://en.cppreference.com/cpp/language/escape)
* [Standard C++ (szkic): lex.literal](https://eel.is/c++draft/lex.literal.kinds) — definicje wszystkich literałów
* [Microsoft Learn: String and character literals](https://learn.microsoft.com/en-us/cpp/cpp/string-and-character-literals-cpp)
  — po polsku: [Literały ciągów i znaków](https://learn.microsoft.com/pl-pl/cpp/cpp/string-and-character-literals-cpp#raw-string-literals-c11)
  (tłumaczenie maszynowe; akapit o UTF-8 jest przetłumaczony błędnie)
* [Microsoft Learn: Literały liczbowe, logiczne i wskaźnikowe](https://learn.microsoft.com/pl-pl/cpp/cpp/numeric-boolean-and-pointer-literals-cpp)
  (po polsku, tłumaczenie maszynowe; typy literałów opisuje dla MSVC)
* [cppreference: std::chrono::duration — literały](https://en.cppreference.com/cpp/chrono/duration#Literals)

---

[Strona główna](../_start.ipynb)